# IMDb Sentiment Analysis
### Web Scraping → Preprocessing → BERT Fine-tuning → Multi-Model Comparison

**Author:** Houcem Hammami
**Dataset:** IMDb reviews scraped via Selenium (Inception — tt1375666)
**Models:** Logistic Regression · SVM · CNN · LSTM · DistilBERT fine-tuned
**Environment:** Google Colab (GPU recommended) or local Python 3.11+

---
**Pipeline overview:**
1. Scrape IMDb reviews with Selenium
2. Preprocess text (tokenise, lemmatise, stop-words)
3. Label sentiment with a zero-shot DistilBERT pipeline
4. Balance classes via contextual word-embedding augmentation
5. Train & compare four classical / deep-learning baselines
6. Fine-tune DistilBERT end-to-end
7. Visualise results and export a reusable inference function

## 1 · Configuration
*Edit this cell only — all paths and hyper-parameters are defined here.*

In [ ]:
# ── Runtime detection ─────────────────────────────────────────────────────────
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
if IN_COLAB:
    drive.mount("/content/drive")
    DATA_DIR   = Path("/content/drive/MyDrive/IMDb_Sentiment_Analysis")
    OUTPUT_DIR = DATA_DIR
else:
    DATA_DIR   = Path("data")
    OUTPUT_DIR = Path("outputs")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── IMDb target ───────────────────────────────────────────────────────────────
MOVIE_ID    = "tt1375666"   # Inception
MAX_CLICKS  = 50            # "Load more" button clicks

# ── Text preprocessing ────────────────────────────────────────────────────────
REVIEW_LANG = "english"     # NLTK stop-words language

# ── Labelling model ───────────────────────────────────────────────────────────
LABELLING_MODEL  = "distilbert-base-uncased-finetuned-sst-2-english"
NEUTRAL_THRESHOLD = 0.80    # scores below this become "neutre"

# ── Augmentation ──────────────────────────────────────────────────────────────
AUGMENT_MODEL = "distilbert-base-uncased"

# ── Classical ML ──────────────────────────────────────────────────────────────
TFIDF_MAX_FEATURES = 5_000
TFIDF_NGRAM_RANGE  = (1, 2)

# ── Deep learning (CNN / LSTM) ────────────────────────────────────────────────
MAX_WORDS  = 10_000
MAX_SEQ_LEN = 200
EMBED_DIM  = 128
DL_EPOCHS  = 10
DL_BATCH   = 32

# ── BERT fine-tuning ──────────────────────────────────────────────────────────
BERT_BASE_MODEL  = "distilbert-base-uncased"
BERT_LR          = 2e-5
BERT_EPOCHS      = 4
BERT_BATCH       = 16
BERT_MAX_LEN     = 256
BERT_OUTPUT_DIR  = OUTPUT_DIR / "bert_finetuned_model"

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42

import os, random, numpy as np
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"]     = "disabled"
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Runtime : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Seed    : {RANDOM_SEED}")

## 2 · Install & Import Dependencies

In [ ]:
# Install in quiet mode to keep output clean
import subprocess, sys

pkgs = [
    "requests", "beautifulsoup4", "selenium",
    "nltk", "nlpaug",
    "torch", "transformers", "datasets", "peft",
    "tensorflow",
    "scikit-learn", "scipy",
    "matplotlib", "seaborn", "wordcloud",
    "tqdm",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    check=True,
)

# Install Chromium on Colab
if IN_COLAB:
    subprocess.run(["apt-get", "install", "-y", "-q", "chromium-browser", "chromium-chromedriver"], check=True)

print("✓ All dependencies installed")

In [ ]:
import os, re, pickle, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import nltk

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample
from wordcloud import WordCloud

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, pipeline,
)
from datasets import Dataset

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"✓ Imports complete | GPU: {torch.cuda.is_available()}")

## 3 · Data Collection — IMDb Scraping

Reviews are scraped from IMDb using Selenium with headless Chromium.
If a cached CSV already exists in `DATA_DIR`, scraping is skipped.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

RAW_CSV = DATA_DIR / "imdb_reviews.csv"


def build_driver() -> webdriver.Chrome:
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_imdb_reviews(movie_id: str, max_clicks: int = MAX_CLICKS) -> pd.DataFrame:
    driver = build_driver()
    url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    print(f"Scraping: {url}")
    driver.get(url)
    time.sleep(2)

    load_selectors = [
        "button.ipc-btn--load-more",
        ".ipc-see-more__button",
        "#load-more-trigger",
        "button[class*='load-more']",
    ]

    for click in range(max_clicks):
        btn = None
        for sel in load_selectors:
            try:
                btn = driver.find_element(By.CSS_SELECTOR, sel)
                break
            except Exception:
                continue
        if btn is None:
            print(f"  No more reviews after {click} clicks")
            break
        driver.execute_script("arguments[0].scrollIntoView({block:'center'})", btn)
        time.sleep(0.5)
        btn.click()
        time.sleep(1.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    containers = soup.select(".review-container") or soup.select("div[class*='review']")
    print(f"  Found {len(containers)} review containers")

    rows = []
    for c in containers:
        title = (c.select_one(".title") or c.select_one("a[class*='title']"))
        text  = (c.select_one(".text.show-more__control") or c.select_one("div[class*='text']"))
        rating_el = c.select_one(".rating-other-user-rating span")
        date_el   = c.select_one(".review-date")

        review_text = text.get_text(strip=True) if text else None
        if not review_text:
            continue

        rating = None
        if rating_el:
            try:
                rating = float(rating_el.get_text(strip=True).split("/")[0])
            except ValueError:
                pass

        rows.append({
            "title":  title.get_text(strip=True) if title else None,
            "review": review_text,
            "rating": rating,
            "date":   date_el.get_text(strip=True) if date_el else None,
        })

    return pd.DataFrame(rows)


# ── Run or load ───────────────────────────────────────────────────────────────
if RAW_CSV.exists():
    df_raw = pd.read_csv(RAW_CSV)
    print(f"✓ Loaded cached data: {len(df_raw)} reviews")
else:
    df_raw = scrape_imdb_reviews(MOVIE_ID, MAX_CLICKS)
    df_raw.to_csv(RAW_CSV, index=False)
    print(f"✓ Scraped & saved: {len(df_raw)} reviews → {RAW_CSV}")

print(df_raw.head(3))

## 4 · Text Preprocessing

- Lowercase & punctuation removal
- Stop-word filtering (NLTK English)
- WordNet lemmatisation

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

_stop  = set(stopwords.words(REVIEW_LANG))
_lemma = WordNetLemmatizer()

PREPROCESSED_CSV = DATA_DIR / "imdb_reviews_preprocessed.csv"


def preprocess(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [_lemma.lemmatize(w) for w in tokens if w not in _stop]
    return " ".join(tokens)


if PREPROCESSED_CSV.exists():
    df_pre = pd.read_csv(PREPROCESSED_CSV)
    print(f"✓ Loaded preprocessed data: {len(df_pre)} rows")
else:
    df_pre = df_raw.copy()
    df_pre["processed_review"] = df_pre["review"].fillna("").astype(str).apply(preprocess)
    df_pre.to_csv(PREPROCESSED_CSV, index=False)
    print(f"✓ Preprocessed & saved → {PREPROCESSED_CSV}")

print(df_pre[["review", "processed_review"]].head(3))

## 5 · Sentiment Labelling (DistilBERT Zero-shot)

A pre-trained DistilBERT SST-2 pipeline assigns coarse labels.
Predictions below `NEUTRAL_THRESHOLD` confidence are marked *neutre*.

In [ ]:
SENTIMENT_CSV = DATA_DIR / "imdb_reviews_with_sentiment.csv"

if SENTIMENT_CSV.exists():
    df_labeled = pd.read_csv(SENTIMENT_CSV)
    print(f"✓ Loaded labelled data: {len(df_labeled)} rows")
    print(df_labeled["sentiment"].value_counts())
else:
    labeller = pipeline(
        "sentiment-analysis",
        model=LABELLING_MODEL,
        device=DEVICE,
        truncation=True,
        max_length=512,
    )

    def label_sentiment(text: str) -> str:
        if not isinstance(text, str) or not text.strip():
            return "neutre"
        result = labeller(text[:512])[0]
        if result["score"] < NEUTRAL_THRESHOLD:
            return "neutre"
        return "positif" if result["label"] == "POSITIVE" else "négatif"

    df_labeled = df_pre.copy()
    df_labeled["sentiment"] = df_labeled["processed_review"].apply(label_sentiment)
    df_labeled.to_csv(SENTIMENT_CSV, index=False)
    print(f"✓ Labelled & saved → {SENTIMENT_CSV}")
    print(df_labeled["sentiment"].value_counts())

## 6 · Class Balancing — Contextual Augmentation

`nlpaug.ContextualWordEmbsAug` generates paraphrases using DistilBERT
to upsample minority classes to match the majority class size.

In [ ]:
import nlpaug.augmenter.word as naw

BALANCED_CSV = DATA_DIR / "imdb_reviews_balanced.csv"


def augment_to_target(df_class: pd.DataFrame, target: int) -> pd.DataFrame:
    label   = df_class["sentiment"].iloc[0]
    current = len(df_class)
    if current >= target:
        return df_class[["processed_review", "sentiment"]].copy()

    print(f"  Augmenting '{label}': {current} → {target}")
    aug = naw.ContextualWordEmbsAug(
        model_path=AUGMENT_MODEL,
        action="substitute",
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    texts  = df_class["processed_review"].tolist()
    needed = target - current
    extras = []

    for t in texts * ((needed // current) + 2):
        if len(extras) >= needed:
            break
        if not t.strip() or len(t.split()) < 3:
            continue
        try:
            aug_t = aug.augment(t)
            aug_t = aug_t[0] if isinstance(aug_t, list) else aug_t
            if aug_t and aug_t.strip() != t:
                extras.append(aug_t)
        except Exception:
            pass

    return pd.DataFrame({
        "processed_review": texts + extras[:needed],
        "sentiment": [label] * (current + len(extras[:needed])),
    })


if BALANCED_CSV.exists():
    df_balanced = pd.read_csv(BALANCED_CSV)
    print(f"✓ Loaded balanced data: {len(df_balanced)} rows")
else:
    target_n = df_labeled["sentiment"].value_counts().max()
    parts    = [augment_to_target(df_labeled[df_labeled["sentiment"] == s], target_n)
                for s in df_labeled["sentiment"].unique()]
    df_balanced = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    df_balanced.to_csv(BALANCED_CSV, index=False)
    print(f"✓ Balanced & saved → {BALANCED_CSV}")

print(df_balanced["sentiment"].value_counts())

## 7 · Classical Models — Logistic Regression & SVM

TF-IDF (bigrams, 5 000 features) → Logistic Regression · Linear SVM

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
X = df_balanced["processed_review"].values
y = df_balanced["sentiment"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y,
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ── TF-IDF ────────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# ── Logistic Regression ───────────────────────────────────────────────────────
logreg = LogisticRegression(max_iter=1_000, random_state=RANDOM_SEED)
logreg.fit(X_train_tfidf, y_train)
y_pred_logreg = logreg.predict(X_test_tfidf)
acc_logreg    = accuracy_score(y_test, y_pred_logreg)
print(f"\nLogistic Regression accuracy: {acc_logreg:.4f}")
print(classification_report(y_test, y_pred_logreg))

# ── SVM ───────────────────────────────────────────────────────────────────────
svm = SVC(kernel="linear", C=1.0, random_state=RANDOM_SEED)
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)
acc_svm    = accuracy_score(y_test, y_pred_svm)
print(f"SVM accuracy: {acc_svm:.4f}")
print(classification_report(y_test, y_pred_svm))

# ── Persist ───────────────────────────────────────────────────────────────────
joblib.dump(tfidf,  OUTPUT_DIR / "tfidf_vectorizer.pkl")
joblib.dump(logreg, OUTPUT_DIR / "logreg_model.pkl")
joblib.dump(svm,    OUTPUT_DIR / "svm_model.pkl")
print("✓ Classical models saved")

## 8 · Deep Learning — CNN & LSTM

Keras embedding → 1D-CNN with global max-pool · Bidirectional LSTM

In [ ]:
# ── Sequence tokenisation ─────────────────────────────────────────────────────
label_names  = sorted(df_balanced["sentiment"].unique())
label_mapping = {lbl: idx for idx, lbl in enumerate(label_names)}

keras_tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
keras_tok.fit_on_texts(X_train)

def to_padded(texts):
    seqs = keras_tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")

X_train_pad = to_padded(X_train)
X_test_pad  = to_padded(X_test)

y_train_num    = np.array([label_mapping[l] for l in y_train])
y_test_num     = np.array([label_mapping[l] for l in y_test])
y_train_onehot = to_categorical(y_train_num, num_classes=len(label_names))
y_test_onehot  = to_categorical(y_test_num,  num_classes=len(label_names))

# ── CNN ───────────────────────────────────────────────────────────────────────
cnn = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_SEQ_LEN),
    Conv1D(128, 5, activation="relu"),
    GlobalMaxPooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(len(label_names), activation="softmax"),
])
cnn.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
cnn.summary()

hist_cnn = cnn.fit(
    X_train_pad, y_train_onehot,
    epochs=DL_EPOCHS, batch_size=DL_BATCH,
    validation_split=0.2, verbose=1,
)
_, acc_cnn   = cnn.evaluate(X_test_pad, y_test_onehot, verbose=0)
y_pred_cnn   = [label_names[i] for i in np.argmax(cnn.predict(X_test_pad, verbose=0), axis=1)]
print(f"\nCNN accuracy: {acc_cnn:.4f}")
print(classification_report(y_test, y_pred_cnn))
cnn.save(OUTPUT_DIR / "cnn_model.keras")

# ── LSTM ──────────────────────────────────────────────────────────────────────
lstm = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_SEQ_LEN),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(len(label_names), activation="softmax"),
])
lstm.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
lstm.summary()

hist_lstm = lstm.fit(
    X_train_pad, y_train_onehot,
    epochs=DL_EPOCHS, batch_size=DL_BATCH,
    validation_split=0.2, verbose=1,
)
_, acc_lstm = lstm.evaluate(X_test_pad, y_test_onehot, verbose=0)
y_pred_lstm = [label_names[i] for i in np.argmax(lstm.predict(X_test_pad, verbose=0), axis=1)]
print(f"\nLSTM accuracy: {acc_lstm:.4f}")
print(classification_report(y_test, y_pred_lstm))
lstm.save(OUTPUT_DIR / "lstm_model.keras")

with open(OUTPUT_DIR / "keras_tokenizer.pkl", "wb") as f:
    pickle.dump(keras_tok, f)
print("✓ Deep-learning models saved")

## 9 · BERT Fine-tuning — DistilBERT End-to-End

Full fine-tuning of `distilbert-base-uncased` on the augmented, balanced dataset.

In [ ]:
# ── Tokenise for BERT ─────────────────────────────────────────────────────────
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_BASE_MODEL)

le = LabelEncoder()
le.fit(y)
y_train_enc = le.transform(y_train).tolist()
y_test_enc  = le.transform(y_test).tolist()

def tokenize_bert(batch):
    return bert_tokenizer(
        batch, padding="max_length", truncation=True, max_length=BERT_MAX_LEN,
    )

train_ds = Dataset.from_dict({"text": list(map(str, X_train)), "label": y_train_enc})
val_ds   = Dataset.from_dict({"text": list(map(str, X_test)),  "label": y_test_enc})

train_ds = train_ds.map(lambda x: tokenize_bert(x["text"]), batched=True)
val_ds   = val_ds.map(  lambda x: tokenize_bert(x["text"]), batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(  "torch", columns=["input_ids", "attention_mask", "label"])

# ── Model & training args ─────────────────────────────────────────────────────
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_BASE_MODEL, num_labels=len(le.classes_),
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "bert_checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=BERT_LR,
    per_device_train_batch_size=BERT_BATCH,
    per_device_eval_batch_size=BERT_BATCH,
    num_train_epochs=BERT_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    save_total_limit=1,
    seed=RANDOM_SEED,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=bert_tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
bert_eval = trainer.evaluate()
acc_bert  = bert_eval["eval_accuracy"]
print(f"\nBERT fine-tuned accuracy: {acc_bert:.4f}")

trainer.save_model(str(BERT_OUTPUT_DIR))
bert_tokenizer.save_pretrained(str(BERT_OUTPUT_DIR))
print(f"✓ Fine-tuned BERT saved → {BERT_OUTPUT_DIR}")

## 10 · Evaluation & Visualisation

In [ ]:
# ── Accuracy comparison ───────────────────────────────────────────────────────
model_names = ["LogReg", "SVM", "CNN", "LSTM", "DistilBERT"]
accuracies  = [acc_logreg, acc_svm, acc_cnn, acc_lstm, acc_bert]
colours     = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(model_names, accuracies, color=colours, alpha=0.85, edgecolor="black")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Model Accuracy Comparison", fontsize=15, fontweight="bold")
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{acc:.4f}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_comparison.png", dpi=300)
plt.show()

# ── Confusion matrices ────────────────────────────────────────────────────────
preds_map = {
    "LogReg":      y_pred_logreg,
    "SVM":         y_pred_svm,
    "CNN":         y_pred_cnn,
    "LSTM":        y_pred_lstm,
}
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Confusion Matrices", fontsize=16, fontweight="bold")
cmaps = ["Blues", "Greens", "Oranges", "Purples"]
for ax, (name, preds), cmap in zip(axes.flat, preds_map.items(), cmaps):
    cm = confusion_matrix(y_test, preds, labels=label_names)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=300)
plt.show()

# ── Learning curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Learning Curves — CNN & LSTM", fontsize=15, fontweight="bold")
for row, (name, hist) in enumerate([("CNN", hist_cnn), ("LSTM", hist_lstm)]):
    axes[row, 0].plot(hist.history["accuracy"],     label="Train")
    axes[row, 0].plot(hist.history["val_accuracy"], label="Val")
    axes[row, 0].set_title(f"{name} Accuracy"); axes[row, 0].legend()
    axes[row, 1].plot(hist.history["loss"],     color="#e74c3c", label="Train")
    axes[row, 1].plot(hist.history["val_loss"], color="#c0392b", label="Val")
    axes[row, 1].set_title(f"{name} Loss"); axes[row, 1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curves.png", dpi=300)
plt.show()

# ── Word clouds ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Word Clouds by Sentiment Class", fontsize=15, fontweight="bold")
colours_wc = {"positif": "Greens", "neutre": "Blues", "négatif": "Reds"}
for ax, (lbl, cmap) in zip(axes, colours_wc.items()):
    text = " ".join(df_balanced[df_balanced["sentiment"] == lbl]["processed_review"].fillna(""))
    wc   = WordCloud(width=800, height=400, background_color="white",
                     colormap=cmap, max_words=100).generate(text)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    ax.set_title(lbl.capitalize(), fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wordclouds.png", dpi=300)
plt.show()
print("✓ All visualisations saved")

## 11 · Results Summary

In [ ]:
results_df = pd.DataFrame({
    "Model":    model_names,
    "Accuracy": [f"{a:.4f}" for a in accuracies],
})
results_df = results_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
results_df.index += 1
print(results_df.to_string())

results_df.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
print(f"\n✓ Results saved → {OUTPUT_DIR / 'model_results.csv'}")

## 12 · Inference Function

Reusable function that accepts any English review text and returns
the predicted sentiment from all five trained models.

In [ ]:
def predict_all(text: str) -> dict:
    """Return sentiment predictions from all five models."""
    proc = preprocess(text)

    # Classical
    vec          = tfidf.transform([proc])
    pred_logreg  = logreg.predict(vec)[0]
    pred_svm     = svm.predict(vec)[0]

    # Deep learning
    pad          = to_padded([proc])
    pred_cnn     = label_names[np.argmax(cnn.predict(pad, verbose=0), axis=1)[0]]
    pred_lstm    = label_names[np.argmax(lstm.predict(pad, verbose=0), axis=1)[0]]

    # BERT fine-tuned
    bert_inf     = pipeline(
        "text-classification", model=str(BERT_OUTPUT_DIR), device=DEVICE,
        truncation=True, max_length=BERT_MAX_LEN,
    )
    raw_label    = bert_inf(text[:512])[0]["label"]
    # map LABEL_0/1/2 → actual class names
    pred_bert    = le.classes_[int(raw_label.split("_")[-1])]

    return {
        "LogReg":     pred_logreg,
        "SVM":        pred_svm,
        "CNN":        pred_cnn,
        "LSTM":       pred_lstm,
        "DistilBERT": pred_bert,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
examples = [
    "Absolutely breathtaking. One of the best films I have ever seen.",
    "I was bored throughout. The plot made no sense whatsoever.",
    "It was decent. Not great, not terrible — just average.",
]

for review in examples:
    preds = predict_all(review)
    print(f"Review : {review[:60]}...")
    for model, sentiment in preds.items():
        print(f"  {model:<12}: {sentiment}")
    print()